<a href="https://colab.research.google.com/github/PedroFrancelino/A-Magia-dos-juros-compostos-e-da-constru-o-de-riqueza/blob/main/Conhe%C3%A7a_o_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instala as bibliotecas necessárias
!pip install openai-whisper openai gTTS

# Instala o ffmpeg (necessário para o Whisper processar áudio)
!sudo apt update && sudo apt install ffmpeg -y

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 8.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.4 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=34fdb7f4083abf4f4f8e956a8caefc4e450ae63afe9af9026da8fc43c5beb5aa
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 ht

In [3]:
from IPython.display import display, Javascript
from google.colab import output
import base64

def record_audio(filename='input_audio.wav'):
  js = Javascript('''
    async function recordAudio() {
      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = 'Clique para Gravar (5 seg)';
      button.style.background = 'red';
      button.style.color = 'white';
      button.style.padding = '10px';
      document.body.appendChild(div);
      div.appendChild(button);

      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      const recorder = new MediaRecorder(stream);
      const chunks = [];

      return new Promise(resolve => {
        button.onclick = () => {
          recorder.start();
          button.textContent = 'Gravando...';
          setTimeout(() => {
            recorder.stop();
            button.textContent = 'Processando...';
          }, 5000); // Grava por 5 segundos
        };

        recorder.onstop = async () => {
          const blob = new Blob(chunks);
          const reader = new FileReader();
          reader.readAsDataURL(blob);
          reader.onloadend = () => resolve(reader.result);
        };
        recorder.ondataavailable = e => chunks.push(e.data);
      });
    }
  ''')
  display(js)
  data = output.eval_js('recordAudio()')
  binary = base64.b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  print(f"Áudio salvo como {filename}")

# Executa a gravação
record_audio()

<IPython.core.display.Javascript object>

Áudio salvo como input_audio.wav


In [4]:
import whisper

# Carrega o modelo (o 'base' é ótimo para velocidade no Colab)
model = whisper.load_model("base")
result = model.transcribe("input_audio.wav")
user_text = result["text"]

print(f"Texto extraído: {user_text}")

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 134MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Texto extraído:  Me diga o que é o chatgpt.


In [7]:
from openai import OpenAI
from google.colab import userdata

# O comando 'userdata.get' busca a chave que você salvou no ícone da tranca
# Isso impede que sua chave apareça no texto do código quando for para o GitHub
minha_chave = userdata.get('OPENAI_API_KEY')

client = OpenAI(api_key=minha_chave)

# Enviando o texto que o Whisper transcreveu para o ChatGPT
completion = client.chat.completions.create(
  model="gpt-3.5-turbo", # Ou "gpt-4" se você tiver acesso
  messages=[
      {"role": "system", "content": "Você é um assistente de voz útil e conciso."},
      {"role": "user", "content": user_text} # 'user_text' vem da etapa do Whisper
  ]
)

# Capturando a resposta em texto
chat_response = completion.choices[0].message.content
print(f"ChatGPT respondeu: {chat_response}")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
from gtts import gTTS
from IPython.display import Audio

# Gera o áudio em português
tts = gTTS(chat_response, lang='pt')
tts.save('response.mp3')

# Exibe o player de áudio com autoplay
display(Audio('response.mp3', autoplay=True))